In [2]:
!pip install faker


In [3]:
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta
import random

fake = Faker()
np.random.seed(42)
random.seed(42)

# ========== 1. 生成用户表 ==========
n_users = 1000
users = pd.DataFrame({
    'user_id': range(1, n_users + 1),
    'register_date': [fake.date_between(start_date='-6M', end_date='today') for _ in range(n_users)],
    'channel': np.random.choice(['organic', 'paid', 'social'], n_users, p=[0.5, 0.3, 0.2])
})
users['register_date'] = pd.to_datetime(users['register_date'])

# ========== 2. 生成行为日志 ==========
events = []
event_types = ['page_view', 'add_to_cart', 'order', 'payment']

for _, user in users.iterrows():
    # 该用户活跃天数随机 1~30 天
    active_days = np.random.randint(1, 31)
    # 从注册日期开始，随机选 active_days 个不同偏移量
    day_offsets = sorted(random.sample(range(0, 180), active_days))  # 半年内
    
    for offset in day_offsets:
        event_date = user['register_date'] + timedelta(days=offset)
        n_events = np.random.randint(1, 11)  # 这天做几件事
        
        current_type = 'page_view'  # 每天第一件事总是浏览
        for _ in range(n_events):
            hour = np.random.randint(0, 24)
            minute = np.random.randint(0, 60)
            second = np.random.randint(0, 60)
            event_time = event_date.replace(hour=hour, minute=minute, second=second)
            
            events.append({
                'user_id': user['user_id'],
                'event_type': current_type,
                'event_time': event_time,
                'product_id': np.random.randint(100, 200)
            })
            
            # 状态机：决定下一个动作
            if current_type == 'page_view':
                if random.random() < 0.3:
                    current_type = 'add_to_cart'
            elif current_type == 'add_to_cart':
                if random.random() < 0.4:
                    current_type = 'order'
                else:
                    current_type = 'page_view'
            elif current_type == 'order':
                if random.random() < 0.7:
                    current_type = 'payment'
                else:
                    current_type = 'page_view'
            else:  # payment 后多半继续逛
                current_type = 'page_view'

# 注入一些脏数据（用于清洗练习）
events_df = pd.DataFrame(events)
# 随机复制一些行作为重复记录
duplicates = events_df.sample(frac=0.02)
events_df = pd.concat([events_df, duplicates], ignore_index=True)
# 随机制造几个未来时间
future_idx = np.random.choice(events_df.index, 10, replace=False)
events_df.loc[future_idx, 'event_time'] = events_df.loc[future_idx, 'event_time'] + timedelta(days=500)
# 制造几个孤儿事件（user_id 不存在）
events_df.loc[events_df.sample(5).index, 'user_id'] = 99999

# 排序并保存
events_df = events_df.sort_values(['user_id', 'event_time'])
users.to_csv('users.csv', index=False)
events_df.to_csv('events.csv', index=False)

print("数据已生成：users.csv 和 events.csv")
print(f"用户数：{len(users)}，事件数：{len(events_df)}")

D:\pythonanaconda\lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


数据已生成：users.csv 和 events.csv
用户数：1000，事件数：87527


In [4]:
import sqlite3
import pandas as pd

# 读取刚才生成的 CSV
users = pd.read_csv('users.csv', parse_dates=['register_date'])
events = pd.read_csv('events.csv', parse_dates=['event_time'])

# 连接 SQLite 数据库（文件 ecommerce.db 会自动创建）
conn = sqlite3.connect('ecommerce.db')

# 写入数据库
users.to_sql('users', conn, if_exists='replace', index=False)
events.to_sql('events', conn, if_exists='replace', index=False)

print("数据库 ecommerce.db 创建成功！")
print(f"users 表行数：{len(users)}")
print(f"events 表行数：{len(events)}")

数据库 ecommerce.db 创建成功！
users 表行数：1000
events 表行数：87527


In [17]:
query = '''
SELECT * FROM events LIMIT 20;
'''
pd.read_sql(query, conn)

,user_id,event_type,event_time,product_id
0,1,order,2026-03-23 05:07:24,181
1,1,page_view,2026-03-23 06:13:14,106
2,1,page_view,2026-03-23 08:44:25,146
3,1,page_view,2026-03-23 09:15:06,116
4,1,payment,2026-03-23 11:14:58,125
5,1,add_to_cart,2026-03-23 16:22:36,179
6,1,page_view,2026-03-23 18:07:30,120
7,1,page_view,2026-03-23 22:25:20,185
8,1,page_view,2026-03-24 09:38:16,113
9,1,page_view,2026-03-24 10:39:25,154


In [6]:
query = '''
SELECT * FROM users LIMIT 5;
'''
pd.read_sql(query, conn)

,user_id,register_date,channel
0,1,2026-03-17 00:00:00,organic
1,2,2026-05-18 00:00:00,social
2,3,2026-05-11 00:00:00,paid
3,4,2026-01-10 00:00:00,paid
4,5,2026-02-21 00:00:00,organic


In [ ]:
#第一步练习：日活跃用户数（DAU）
#业务含义：每天有多少独立用户发生了行为。
#SQL 思路：
#想要一张表，列是日期 + 用户数。
#数据在 events 表，有 event_time 和 user_id。
#截取日期 → 按日期分组 → 每组的去重用户数。

In [16]:
query_dau = '''
SELECT date(event_time) as 日期 ,
count(DISTINCT(user_id)) as 活跃人数 
FROM events 
group by 日期
order by 日期
LIMIT 10;
'''
pd.read_sql(query_dau, conn)

,日期,活跃人数
0,2025-12-03,1
1,2025-12-04,2
2,2025-12-05,2
3,2025-12-06,6
4,2025-12-07,3
5,2025-12-08,4
6,2025-12-09,3
7,2025-12-10,8
8,2025-12-11,6
9,2025-12-12,2


In [27]:
#两列——event_type（事件类型）和 uv（独立用户数），每种事件一行。
#数据来源：events 表。
#步骤：
   #只保留漏斗的四种事件类型。
   #限定时间范围为 5 月。
   #按 event_type 分组，每组计算去重用户数。
   #按人数降序排列（浏览最多，支付最少，自然形成漏斗形状）。

In [28]:
query_check_date = '''
SELECT MIN(DATE(event_time)) AS 最早日期,
       MAX(DATE(event_time)) AS 最晚日期
FROM events;
'''
pd.read_sql(query_check_date, conn)

,最早日期,最晚日期
0,2025-12-03,2027-12-28


In [26]:
query_uv = '''
SELECT event_type as 事件类型 ,
count(DISTINCT(user_id)) as 独立用户数 
FROM events 
where event_type in ('page_view', 'add_to_cart', 'order', 'payment')
and date(event_time) between '2026-01-03'and'2026-02-03'
group by event_type
order by 独立用户数 desc
LIMIT 4;
'''
pd.read_sql(query_uv, conn)
query_uv = '''


,事件类型,独立用户数
0,page_view,248
1,add_to_cart,220
2,order,132
3,payment,91


In [30]:
# 把刚才的漏斗 DataFrame 导出为 CSV
funnel_df = pd.read_sql(query_uv, conn)   # query_uv 是你刚才写的漏斗查询
funnel_df.to_csv('funnel.csv', index=False)
print("漏斗数据已导出为 funnel.csv")

漏斗数据已导出为 funnel.csv


In [34]:
# 留存基表（每个用户首次/末次活跃日期）
query_retention_base = '''
SELECT user_id,
       MIN(DATE(event_time)) AS first_active,
       MAX(DATE(event_time)) AS last_active
FROM events
GROUP BY user_id;
'''
retention_df = pd.read_sql(query_retention_base, conn)
pd.read_sql(query_retention_base, conn)

,user_id,first_active,last_active
0,1,2026-03-23,2026-09-06
1,2,2026-05-30,2026-11-08
2,3,2026-05-11,2026-10-31
3,4,2026-01-10,2026-07-04
4,5,2026-02-22,2026-08-18
...,...,...,...
996,997,2026-05-19,2026-11-11
997,998,2026-04-22,2026-10-05
998,999,2026-05-11,2026-10-09
999,1000,2026-06-28,2026-06-28


In [39]:
query_dau = '''
SELECT DATE(event_time) AS 事件日期,
       COUNT(DISTINCT user_id) AS 独立用户数
FROM events
WHERE event_type IN ('page_view', 'add_to_cart', 'order', 'payment')
  AND DATE(event_time) BETWEEN '2026-01-03' AND '2026-02-03'
GROUP BY DATE(event_time)
ORDER BY DATE(event_time);
'''
pd.read_sql(query_dau, conn)

,事件日期,独立用户数
0,2026-01-03,11
1,2026-01-04,15
2,2026-01-05,12
3,2026-01-06,16
4,2026-01-07,22
5,2026-01-08,12
6,2026-01-09,19
7,2026-01-10,21
8,2026-01-11,19
9,2026-01-12,30


In [41]:
import pandas as pd
import numpy as np
from datetime import datetime

# 加载数据
users = pd.read_csv('users.csv', parse_dates=['register_date'])
events = pd.read_csv('events.csv', parse_dates=['event_time'])

# 看看用户表大小
print('用户表大小：', users.shape)
print(users.head(3))

# 看看事件表大小
print('\n事件表大小：', events.shape)
print(events.head(3))

用户表大小： (1000, 3)
   user_id register_date  channel
0        1    2026-03-17  organic
1        2    2026-05-18   social
2        3    2026-05-11     paid

事件表大小： (87527, 4)
   user_id event_type          event_time  product_id
0        1      order 2026-03-23 05:07:24         181
1        1  page_view 2026-03-23 06:13:14         106
2        1  page_view 2026-03-23 08:44:25         146


In [42]:
duplicate_rows = events.duplicated()
print('重复记录数：', duplicate_rows.sum())
future_events = events[events['event_time'] > pd.Timestamp.now()]
print('未来事件数：', len(future_events))
print(future_events[['user_id', 'event_time']].head())
orphan_events = events[~events['user_id'].isin(users['user_id'])]
print('孤儿事件数：', len(orphan_events))
print(orphan_events['user_id'].unique())

重复记录数： 1716
未来事件数： 43153
    user_id          event_time
64        1 2026-07-03 01:13:12
65        1 2026-07-03 06:04:42
66        1 2026-07-03 06:30:17
67        1 2026-07-03 09:31:42
68        1 2026-07-03 11:38:01
孤儿事件数： 5
[99999]


In [43]:
# 备份原始数据（养成好习惯）
events_clean = events.copy()

# 1. 删除重复行
events_clean = events_clean.drop_duplicates()
print('去重后事件数：', len(events_clean))

# 2. 删除未来时间
events_clean = events_clean[events_clean['event_time'] <= pd.Timestamp.now()]
print('去除未来时间后事件数：', len(events_clean))

# 3. 删除孤儿事件（只保留有效用户）
events_clean = events_clean[events_clean['user_id'].isin(users['user_id'])]
print('去除孤儿事件后事件数：', len(events_clean))

去重后事件数： 85811
去除未来时间后事件数： 43495
去除孤儿事件后事件数： 43493


In [44]:
# 用户活跃天数
user_active_days = events_clean.groupby('user_id')['event_time'].apply(
    lambda x: x.dt.date.nunique()
).reset_index(name='active_days')

# 用户各类事件次数
user_events_cnt = events_clean.groupby('user_id')['event_type'].value_counts().unstack(fill_value=0).reset_index()
user_events_cnt.columns = ['user_id', 'add_to_cart', 'order', 'page_view', 'payment']  # 确保列名对齐

# 用户总事件数
user_total_events = events_clean.groupby('user_id').size().reset_index(name='total_events')

# 合并所有特征
user_features = users.merge(user_total_events, on='user_id', how='left')
user_features = user_features.merge(user_active_days, on='user_id', how='left')
user_features = user_features.merge(user_events_cnt, on='user_id', how='left')

# 把缺失值填0（比如有些用户没有任何事件）
user_features.fillna(0, inplace=True)

# 看一下特征表
user_features.head()

,user_id,register_date,channel,total_events,active_days,add_to_cart,order,page_view,payment
0,1,2026-03-17,organic,62.0,10.0,14.0,6.0,38.0,4.0
1,2,2026-05-18,social,15.0,2.0,4.0,0.0,11.0,0.0
2,3,2026-05-11,paid,36.0,6.0,7.0,4.0,23.0,2.0
3,4,2026-01-10,paid,49.0,8.0,6.0,4.0,36.0,3.0
4,5,2026-02-21,organic,89.0,13.0,18.0,4.0,65.0,2.0


In [45]:
def segment(row):
    if row['payment'] > 0 and row['active_days'] >= 5:
        return '高价值客户'
    elif row['payment'] > 0:
        return '普通付费客户'
    elif row['add_to_cart'] > 0 and row['payment'] == 0:
        return '犹豫中客户'
    elif row['page_view'] >= 10 and row['add_to_cart'] == 0 and row['payment'] == 0:
        return '活跃浏览客户'
    else:
        return '低活客户'

user_features['segment'] = user_features.apply(segment, axis=1)

# 查看分群分布
print(user_features['segment'].value_counts())

segment
高价值客户     489
犹豫中客户     256
低活客户      135
普通付费客户    120
Name: count, dtype: int64


In [46]:
user_features.to_csv('user_segments.csv', index=False)
print("用户分群结果已导出为 user_segments.csv")

用户分群结果已导出为 user_segments.csv


In [47]:
import os
print(os.getcwd())

C:\Users\86180


In [48]:
import pandas as pd
import sqlite3

# 连接数据库
conn = sqlite3.connect('ecommerce.db')

# 生成用户每周活跃标记
query_weekly_active = '''
SELECT user_id,
       strftime('%Y-%W', event_time) AS active_week
FROM events
WHERE event_time <= datetime('now')
  AND user_id IN (SELECT user_id FROM users)
GROUP BY user_id, active_week
ORDER BY user_id, active_week;
'''

weekly_active_df = pd.read_sql(query_weekly_active, conn)
weekly_active_df.to_csv('user_weekly_activity.csv', index=False)

print("导出成功！")
print(f"文件：user_weekly_activity.csv，行数：{len(weekly_active_df)}")

导出成功！
文件：user_weekly_activity.csv，行数：5774


In [49]:
import pandas as pd
import sqlite3

# 连接数据库
conn = sqlite3.connect('ecommerce.db')

# 读取全量事件（只取正常时间）
events = pd.read_sql('''
    SELECT user_id, event_time
    FROM events
    WHERE event_time <= datetime('now')
      AND user_id IN (SELECT user_id FROM users)
''', conn, parse_dates=['event_time'])

# 将事件时间转换为周标识 (年-周，周一为每周第一天)
events['active_week'] = events['event_time'].dt.strftime('%Y-%W')

# 每个用户每周只保留一条记录（活跃过就算）
weekly_active = events[['user_id', 'active_week']].drop_duplicates()

# 计算每个用户的首次活跃周
first_week = weekly_active.groupby('user_id')['active_week'].min().reset_index()
first_week.columns = ['user_id', 'first_week']

# 将首次活跃周合并回周活明细
weekly_with_cohort = weekly_active.merge(first_week, on='user_id')

# 计算相对周数（第几周）
# 将周标识转换为 ordinal 数值（便于减法）
weekly_with_cohort['active_week_ord'] = weekly_with_cohort['active_week'].str.replace('-','').astype(int)
weekly_with_cohort['first_week_ord'] = weekly_with_cohort['first_week'].str.replace('-','').astype(int)
# 注意：由于 '2026-05' 这种格式不能直接减，需要转换成日期。更稳健的方法：
# 将周标识转为该周的周一日期
weekly_with_cohort['active_week_date'] = pd.to_datetime(weekly_with_cohort['active_week'] + '-1', format='%Y-%W-%w')
weekly_with_cohort['first_week_date'] = pd.to_datetime(weekly_with_cohort['first_week'] + '-1', format='%Y-%W-%w')
# 计算相差的周数
weekly_with_cohort['week_number'] = (weekly_with_cohort['active_week_date'] - weekly_with_cohort['first_week_date']).dt.days // 7

# 统计每个队列在每个相对周的用户数
cohort_counts = weekly_with_cohort.groupby(['first_week', 'week_number'])['user_id'].nunique().unstack(fill_value=0)

# 计算留存率：每个队列第N周人数 / 该队列首周人数
first_week_sizes = cohort_counts[0]  # 每周队列的新增用户数
cohort_retention = cohort_counts.div(first_week_sizes, axis=0)

# 显示留存矩阵（绝对人数）
print("=== 周留存人数矩阵（示例前5个队列） ===")
print(cohort_counts.head())

# 导出两个矩阵
cohort_counts.to_csv('retention_counts.csv')
cohort_retention.to_csv('retention_rates.csv')

print("\n留存矩阵已导出：retention_counts.csv (人数) 和 retention_rates.csv (留存率)")

=== 周留存人数矩阵（示例前5个队列） ===
week_number  0   1   2   3   4   5   6   7   8   9   ...  17  18  19  20  21  \
first_week                                           ...                       
2025-48      11   6   3   6   7   3   6   8  10   5  ...   5   5   3   7   5   
2025-49      24  12  13   9  12  12  16  14  10  14  ...  12  11  15   9   9   
2025-50      21  11  11  11  15  13   9   9  10   9  ...  12  12  13  16   7   
2025-51      25  14  17  13  13  16  14  10  16  12  ...  11  12  11  17  14   
2025-52      10   3   4   3   3   5   5   5   1   1  ...   6   1   5   4   2   

week_number  22  23  24  25  26  
first_week                       
2025-48       4   7   5   6   1  
2025-49       7  12  12   4   0  
2025-50      14  12   5   0   0  
2025-51      13   3   0   0   0  
2025-52       3   0   0   0   0  

[5 rows x 27 columns]

留存矩阵已导出：retention_counts.csv (人数) 和 retention_rates.csv (留存率)


In [50]:
# 将留存率矩阵转为长格式（接前面的 DataFrame cohort_retention）
cohort_retention_long = cohort_retention.reset_index().melt(id_vars='first_week', var_name='week_number', value_name='retention_rate')
cohort_retention_long.to_csv('retention_rates_long.csv', index=False)
print("长格式留存数据已导出")


长格式留存数据已导出
